# Ingestão Bronze — Download INEP + Validação

Este notebook executa a ingestão dos Microdados do Censo Escolar diretamente do servidor do INEP e valida a camada Bronze resultante.

**Não requer arquivos locais.** Os ZIPs são baixados automaticamente de `download.inep.gov.br`.

> **Pré-requisito:** Configure as variáveis de ambiente `BRONZE_DATA_PATH`, `SILVER_DATA_PATH` e `GOLD_DATA_PATH` no cluster antes de executar.


## Passo 1: Configuração do ambiente

In [ ]:
import sys
import os
from pathlib import Path

# Adiciona a raiz do projeto ao sys.path
# Ajuste o caminho abaixo para o seu usuário do Databricks
PROJECT_ROOT = "/Workspace/Repos/SEU_EMAIL/ods4-data-lakehouse-inep"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Opcional: sobrescrever caminhos via variável (se não estiver no cluster)
# os.environ["BRONZE_DATA_PATH"] = "/dbfs/FileStore/lakehouse_inep/bronze"

from src.config import settings
from src.utils.logger import get_logger

logger = get_logger("notebook_bronze")
print(f"Bronze path: {settings.BRONZE_DATA_PATH}")
print("✓ Ambiente configurado")

## Passo 2: Download dos dados do INEP (Web → Bronze)

Baixa os ZIPs diretamente de `download.inep.gov.br` e extrai os CSVs para a camada Bronze.

- Anos cobertos: **2004–2024** (exceto 2020 — sem publicação pelo INEP)
- Anos já baixados são **pulados automaticamente**

> ⚠️ Este processo pode demorar **2–4 horas** dependendo da rede do cluster.

In [ ]:
from src.jobs.bronze.ingest_from_web import run as ingest_bronze

# Para baixar todos os anos (2004–2024):
ingest_bronze()

# Para baixar um intervalo específico:
# ingest_bronze(ano_inicio=2020, ano_fim=2024)

# Para forçar re-download de anos já existentes:
# ingest_bronze(forcar_redownload=True)

## Passo 3: Validação da Camada Bronze

In [ ]:
import pandas as pd

bronze_path = Path(settings.BRONZE_DATA_PATH)
ano_dirs = sorted(bronze_path.glob("ano=*"))

print(f"{'Ano':<10} {'Arquivos CSV':>14} {'Tamanho (MB)':>14}")
print("-" * 42)
total_mb = 0
for d in ano_dirs:
    csvs = list(d.glob("*.csv"))
    mb = sum(f.stat().st_size for f in csvs) / 1_048_576
    total_mb += mb
    status = "✓" if csvs else "✗ vazio"
    print(f"{d.name:<10} {status:>14} {mb:>13.1f}")
print("-" * 42)
print(f"{'TOTAL':<10} {len(ano_dirs):>14} {total_mb:>13.1f}")

In [ ]:
# Amostra do arquivo mais recente
csv_2024 = bronze_path / "ano=2024" / "microdados_ed_basica_2024.csv"
if csv_2024.exists():
    df = pd.read_csv(csv_2024, sep=";", encoding="latin-1", nrows=3)
    print(f"Colunas: {len(df.columns)} | Linhas (amostra): {len(df)}")
    display(df.head(3))
else:
    print("Arquivo 2024 ainda não disponível.")